In [1]:
import os, warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    module="keras.src.export.tf2onnx_lib"
)
import tensorflow as tf
import pandas as pd
from src.constants import *
from src.helpers import load_img, build_dataset, decode_model_output, load_MY_model, get_encode_funs
import editdistance

In [2]:
df = pd.read_csv(TEST_TSV, sep='\t', header=None, names=['file', 'label'])
df = df.dropna(subset=['label']).reset_index(drop=True)
image_paths = [TEST_DIR + "/" + f for f in df["file"].values]
labels = df["label"].values

In [3]:
label_to_int, label_to_str = get_encode_funs()
model = load_MY_model()
test_ds = build_dataset(image_paths, labels, 32, label_to_int)
pred_y = model.predict(test_ds)

I0000 00:00:1769344581.156154    1539 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 6053 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2080 SUPER, pci bus id: 0000:01:00.0, compute capability: 7.5
/home/silver/tf-venv310/lib/python3.10/site-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: img_input
Received: inputs=['Tensor(shape=(32, 200, 800, 1))']
  warnings.warn(msg)


48/49 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step

/home/silver/tf-venv310/lib/python3.10/site-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: img_input
Received: inputs=['Tensor(shape=(None, 200, 800, 1))']
  warnings.warn(msg)


49/49 ━━━━━━━━━━━━━━━━━━━━ 3s 44ms/step


In [4]:
truths = []
preds = []
preds_probs = []
for imgs, labels in test_ds:
    pred = model(imgs, training=False)
    preds_probs.append(pred)
    preds.extend(decode_model_output(pred, label_to_str))
    for seq in labels.numpy():
        seq = seq[seq != 0]
        truths.append(b"".join(label_to_str(seq).numpy()).decode("utf-8"))

total_edits = 0
total_chars = 0
for p, t in zip(preds, truths):
    total_edits += editdistance.eval(p, t)
    total_chars += len(t)
cer = total_edits / max(1, total_chars)

correct = sum(p == t for p, t in zip(preds, truths))
ewm = correct / max(1, len(truths))

logits = tf.concat(preds_probs, axis=0)
pred_ids = tf.argmax(logits, axis=-1)
blank_idx = logits.shape[-1] - 1
blanks = tf.equal(pred_ids, blank_idx)
blank_cnt = tf.reduce_sum(tf.cast(blanks, tf.float32))
total_steps = tf.cast(tf.size(pred_ids), tf.float32)
blank_ratio = (blank_cnt / tf.maximum(1., total_steps)).numpy()

pred_lengths = [len(p) for p in preds]
truth_lengths = [len(t) for t in truths]
len_ratio = sum(pred_lengths) / max(1, sum(truth_lengths))

print(f"Cer: {cer}")
print(f"Ewm: {ewm}")
print(f"Blank ratio: {blank_ratio}")
print(f"Length ratio: {len_ratio}")

/home/silver/tf-venv310/lib/python3.10/site-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: img_input
Received: inputs=['Tensor(shape=(8, 200, 800, 1))']
  warnings.warn(msg)


Cer: 0.5657335581787521
Ewm: 0.03562176165803109
Blank ratio: 0.8150777220726013
Length ratio: 0.8956492411467116
